# 配套实践 01-02：认识机器人数据的张量形状

本练习对应基础篇第 01 章“机器人眼中的世界是什么”。我们将构造一批简化的图像、机器人本体状态、动作和语言数据，观察 batch、time、channel 等维度的含义。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/basics/01-robot-view-of-the-world/" target="_blank">在新标签页打开课程正文</a>

预计时间：10～15 分钟。不需要 GPU，也不需要下载外部数据。

## 使用方法

按照从上到下的顺序运行单元格。练习使用的图像是随机生成的小数组，只用于解释形状，不包含真实视觉内容。

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
print("环境准备完成")

## 1. 定义一批机器人数据

下面的字母分别表示批次大小、时间长度、图像通道、高度、宽度、本体状态维度、动作维度和语言长度。你可以稍后修改这些数字。

In [ ]:
B = 2
T = 5
C = 3
H = 32
W = 32
P = 10
A = 7
L = 6

images = rng.random((B, T, C, H, W), dtype=np.float32)
proprio = rng.normal(size=(B, T, P)).astype(np.float32)
actions = rng.normal(size=(B, T, A)).astype(np.float32)
instruction_tokens = rng.integers(0, 1000, size=(B, L))

shape_table = pd.DataFrame({
    "数据": ["图像", "本体状态", "动作", "语言编号"],
    "实际形状": [images.shape, proprio.shape, actions.shape, instruction_tokens.shape],
    "维度含义": ["B,T,C,H,W", "B,T,P", "B,T,A", "B,L"],
})

shape_table

四组数据共享同一个批次维度 B。图像、本体状态和动作还共享时间维度 T，这意味着同一个批次位置和时间位置上的数据应当属于同一条轨迹、同一个时刻。

## 2. 取出一条轨迹和一个时间步

In [ ]:
sample_index = 0
time_index = 2

one_trajectory_images = images[sample_index]
one_image = images[sample_index, time_index]
one_proprio = proprio[sample_index, time_index]
one_action = actions[sample_index, time_index]

print("一条图像轨迹：", one_trajectory_images.shape)
print("一个时刻的图像：", one_image.shape)
print("一个时刻的本体状态：", one_proprio.shape)
print("一个时刻的动作：", one_action.shape)

取出批次中的一个样本后，B 维消失；再取出一个时间步后，T 维也消失。剩下的维度描述该时刻具体数据的内部结构。

## 3. 处理不同长度的轨迹

假设第一条轨迹有 5 个有效时间步，第二条只有 3 个。为了组成同一个张量，第二条仍然占用 5 个位置，但最后两个位置只是补齐。

In [ ]:
valid_lengths = np.maximum(T - 2 * np.arange(B), 1)
time_indices = np.arange(T)[None, :]
time_mask = time_indices < valid_lengths[:, None]

mask_table = pd.DataFrame(
    time_mask.astype(int),
    index=[f"轨迹 {i + 1}" for i in range(B)],
    columns=[f"第 {i + 1} 步" for i in range(T)],
)

mask_table

数值 1 表示真实时间步，数值 0 表示补齐位置。模型计算平均值或误差时，不应把补齐位置当成真实数据。

## 4. 比较忽略和使用有效位置标记的结果

In [ ]:
scalar_signal = np.arange(1, B * T + 1, dtype=float).reshape(B, T)
scalar_signal[~time_mask] = 0.0

mean_without_mask = scalar_signal.mean(axis=1)
mean_with_mask = (scalar_signal * time_mask).sum(axis=1) / time_mask.sum(axis=1)

comparison = pd.DataFrame({
    "未使用有效位置标记": mean_without_mask,
    "使用有效位置标记": mean_with_mask,
}, index=[f"轨迹 {i + 1}" for i in range(B)])

comparison

第一条轨迹没有补齐位置，因此两种结果相同。较短轨迹末尾的零只是补齐内容，如果直接参与平均，就会错误地降低结果。

## 5. 自己修改维度

返回第一个参数单元格，尝试进行以下修改，然后重新运行后续单元格：

1. 将 B 从 2 改为 4，观察哪些形状发生变化。
2. 将 T 从 5 改为 8，观察有效长度和补齐位置怎样变化。
3. 将图像尺寸从 32×32 改为 64×64，观察数组大小。
4. 修改 valid_lengths 的生成规则，比较两种平均结果。

## 小结

张量的每个维度都有明确语义。B 表示不同样本，T 表示时间，C、H、W 描述图像结构，P 和 A 分别描述机器人状态与动作。不同长度的轨迹需要补齐，并使用有效位置标记区分真实数据和人为补齐内容。